# Prueba de formato: 3 condiciones con Qwen2.5-1.5B

5 preguntas escritas a mano (`data/prueba.jsonl`; 3 chilenas, 2 no chilenas). Objetivo: revisar a ojo el formato de los prompts, el parseo de las acciones de ReAct y las observaciones. **No** mide desempeño. GPU: T4.

In [ ]:
# Ajustar a donde esté el repo en Colab (clonado o en Drive).
import os
os.chdir('/content/proyecto')
!ls

In [ ]:
# Si vllm no está instalado en esta sesión (versión probada: 0.34.0):
# !pip install vllm==0.34.0

## 1. Ver los prompts sin cargar el modelo (solo el tokenizer)

In [ ]:
from transformers import AutoTokenizer
import json, llm, agents

class SoloTokenizer:
    """Imita a vllm.LLM lo justo para armar prompts sin usar la GPU."""
    def __init__(self, nombre):
        self.tok = AutoTokenizer.from_pretrained(nombre)
    def get_tokenizer(self):
        return self.tok

falso = SoloTokenizer(llm.MODELOS['1.5B'])
q = json.loads(open('data/prueba.jsonl', encoding='utf-8').readline())
usuario = (f"{agents.INSTR_REACT}

Ejemplos:

{agents.EJEMPLOS_REACT}

"
           f"Pregunta: {q['pregunta']}")
p = llm.armar_prompt(falso, agents.SISTEMA, usuario, prefijo_asistente='Pensamiento 1:')
print(p)
print('
Tokens del prompt ReAct (paso 1):', llm.contar_tokens(falso, p))

## 2. Correr las 3 condiciones (proceso aparte, libera la GPU al terminar)

In [ ]:
!python correr.py 1.5B prueba

## 3. Revisar las trazas

In [ ]:
trazas = [json.loads(l) for l in open('resultados/trazas_1.5B_prueba.jsonl', encoding='utf-8')]
for t in trazas:
    print('=' * 100)
    print(f"[{t['condicion']}] {t['pregunta']}")
    print(f"gold: {t['respuesta_gold']} | modelo: {t['respuesta_modelo']!r} | "
          f"termino: {t['termino']} | formato_ok: {t['formato_ok']} | "
          f"acciones inválidas: {t['acciones_invalidas']}")
    if t['titulos_vistos']:
        print('títulos vistos:', t['titulos_vistos'])
    if t['condicion'] == 'react':
        print('-' * 100)
        print(t['texto_modelo'])
    else:
        print('salida cruda:', repr(t['texto_modelo']))

Qué mirar:
- directa/rag: `formato_ok` y respuestas cortas (sin explicación).
- react: que cada paso tenga `Acción k: Buscar[...]` o `Terminar[...]`, que el modelo no invente observaciones, que `Buscar` encuentre páginas (si no, ver los 'Similares'), y cuántos pasos usa. Si aparece `termino='contexto'`, subir `llm.MAX_MODEL_LEN`.

In [ ]:
!python evaluate.py prueba